# Week 63, Day 6: Forge: Week 63 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w063/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", -20.0 - (-23.5))
print("2.", round(1 - 0.8 ** 2, 3))
print("3.", round(float(np.exp(2 * 0.5 + 4 * 0.2 / 2)), 4))

### Exercise: Part A: variational EM for a Poisson latent model

Complete the M step: for each neuron, the gradient of its expected log-likelihood with respect to (c_n, d_n) and the Hessian, using the expected rate mu_tn = exp(c_n m_t + d_n + c_n^2 v_t / 2). Run 25 iterations of variational EM.

In [ ]:
import numpy as np
gA = np.random.default_rng(636)
T, N = 300, 40
t = np.arange(T)
z_true = np.sin(2 * np.pi * t / 120) + 0.6 * np.sin(2 * np.pi * t / 47 + 1.0)
z_true = (z_true - z_true.mean()) / z_true.std()
c_true = gA.normal(0, 0.6, N); d_true = np.log(gA.uniform(0.3, 1.0, N))
X = gA.poisson(np.exp(np.outer(z_true, c_true) + d_true))

a = 0.97
main = np.r_[1.0, np.full(T - 2, 1 + a ** 2), 1.0] / (1 - a ** 2); off = np.full(T - 1, -a / (1 - a ** 2))
Q = np.diag(main) + np.diag(off, 1) + np.diag(off, -1)

def e_step(c, d, m0):
    m = m0.copy()
    for _ in range(20):
        lam = np.exp(np.outer(m, c) + d)
        H = np.diag((lam * c ** 2).sum(1)) + Q
        m = m + np.linalg.solve(H, ((X - lam) * c).sum(1) - Q @ m)
    lam = np.exp(np.outer(m, c) + d)
    v = np.diag(np.linalg.inv(np.diag((lam * c ** 2).sum(1)) + Q))
    return m, v

c_hat = gA.normal(0, 0.1, N); d_hat = np.log(X.mean(0) + 1e-3); m_hat = np.zeros(T)
ell = []
for it in range(25):
    m_hat, v_hat = e_step(c_hat, d_hat, m_hat)
    for n in range(N):
        mu_n = np.exp(c_hat[n] * m_hat + d_hat[n] + c_hat[n] ** 2 * v_hat / 2)
        dmu_dc = mu_n * (m_hat + c_hat[n] * v_hat)
        g_c = np.sum(X[:, n] * m_hat - dmu_dc)
        g_d = ...
        H_cc = -np.sum(mu_n * (m_hat + c_hat[n] * v_hat) ** 2 + mu_n * v_hat)
        H_cd = -np.sum(dmu_dc)
        H_dd = -np.sum(mu_n)
        step = np.linalg.solve(np.array([[H_cc, H_cd], [H_cd, H_dd]]), np.array([g_c, g_d]))
        c_hat[n] -= step[0]; d_hat[n] -= step[1]
    mu = np.exp(np.outer(m_hat, c_hat) + d_hat + np.outer(v_hat, c_hat ** 2) / 2)
    ell.append(np.sum(X * (np.outer(m_hat, c_hat) + d_hat) - mu))
print(f"expected log-likelihood term: {ell[0]:.1f} -> {ell[-1]:.1f}")
print(f"|corr| of loadings with truth {abs(np.corrcoef(c_hat, c_true)[0, 1]):.3f}; of latent with truth {abs(np.corrcoef(m_hat, z_true)[0, 1]):.3f}")

In [ ]:
import numpy as np
def test_loadings():
    "the learned loadings match the truth up to sign and scale (absolute correlation above 0.9)"
    assert abs(np.corrcoef(c_hat, c_true)[0, 1]) > 0.9
def test_latent():
    "the inferred latent matches the true latent (absolute correlation above 0.9)"
    assert abs(np.corrcoef(m_hat, z_true)[0, 1]) > 0.9
def test_elbo():
    "the expected log-likelihood term rises over the EM iterations"
    assert ell[-1] > ell[0]

test_loadings()
test_latent()
test_elbo()
print("All checks passed")

### Exercise: Part B: importance-weighted bounds

For one observation of a 12-dimensional PPCA model, use a deliberately poor Gaussian q (the posterior mean shifted by 0.3 and the covariance doubled). For K in 1, 10, 100 and 1000, estimate L_K by averaging log-mean-exp of K log-weights over 200 repetitions.

In [ ]:
from scipy.stats import multivariate_normal as mvn
from scipy.special import logsumexp
gB = np.random.default_rng(6361)
d, k, s2 = 12, 2, 0.5
W = gB.normal(size=(d, k)); M = W.T @ W + s2 * np.eye(k)
x0 = W @ gB.normal(size=k) + np.sqrt(s2) * gB.normal(size=d)
logev = mvn.logpdf(x0, np.zeros(d), W @ W.T + s2 * np.eye(d))
m_post = np.linalg.solve(M, W.T @ x0); S_post = s2 * np.linalg.inv(M)
mq, Sq = m_post + 0.3, 2 * S_post

def log_w(zs):
    return (mvn.logpdf(x0 - zs @ W.T, np.zeros(d), s2 * np.eye(d)) + mvn.logpdf(zs, np.zeros(k), np.eye(k))
            - mvn.logpdf(zs, mq, Sq))

bounds = []
for K in (1, 10, 100, 1000):
    vals = []
    for rep in range(200):
        lw = log_w(gB.multivariate_normal(mq, Sq, size=K))
        vals.append(...)
    bounds.append(np.mean(vals))
print(f"log evidence {logev:.3f}")
print("bound for K = 1, 10, 100, 1000:", np.round(bounds, 3))

In [ ]:
import numpy as np
def test_monotone():
    "the bound rises with K and stays below the log evidence"
    assert all(b >= a - 0.05 for a, b in zip(bounds, bounds[1:])) and bounds[-1] < logev + 0.05
def test_closer():
    "K = 1000 closes more than half of the gap left by the ELBO"
    assert (logev - bounds[-1]) < 0.5 * (logev - bounds[0])

test_monotone()
test_closer()
print("All checks passed")